# Exercise 1a (DF)
We create an "origin_reference" DataFrame mapping short origin codes to full names.
Then we join with the cars DataFrame to get full origin names.
Finally, we select Car, Origin, FullName, MPG, Weight.


In [ ]:
origin_reference_Tamele70274 = spark_Tamele70274.createDataFrame([
    ("US", "United States"),
    ("Europe", "European Union"),
    ("Japan", "Japan")
], ["Origin", "FullName"])

cars_with_origin_df_Tamele70274 = (
    df_Tamele70274.join(origin_reference_Tamele70274, "Origin", "left")
                  .select("Car", "Origin", "FullName", "MPG", "Weight")
)

cars_with_origin_df_Tamele70274.show(5, truncate=False)


# Exercise 1b (SQL)
We create an "origin_reference" view and join it with the cars table.
We select Car, Origin, FullName, MPG, Weight.


In [ ]:
origin_reference_Tamele70274.createOrReplaceTempView("origin_reference")

query_origin_sql_Tamele70274 = """
SELECT c.Car, c.Origin, o.FullName, c.MPG, c.Weight
FROM cars c
LEFT JOIN origin_reference o
  ON c.Origin = o.Origin
"""

cars_with_origin_sql_Tamele70274 = spark_Tamele70274.sql(query_origin_sql_Tamele70274)
cars_with_origin_sql_Tamele70274.show(5, truncate=False)


# Exercise 2a (DF)
We create "weight_classes" DataFrame with four ranges.
We cross join with cars and filter to assign each car to its weight class.
We select Car, Weight, Class, Horsepower.


In [ ]:
weight_classes_Tamele70274 = spark_Tamele70274.createDataFrame([
    ("Light", 0, 2500),
    ("Medium", 2500, 3500),
    ("Heavy", 3500, 4500),
    ("Very Heavy", 4500, 6000)
], ["Class", "MinWeight", "MaxWeight"])

cars_weight_class_df_Tamele70274 = (
    df_Tamele70274.crossJoin(weight_classes_Tamele70274)
    .filter((col("Weight") >= col("MinWeight")) & (col("Weight") < col("MaxWeight")))
    .select("Car", "Weight", "Class", "Horsepower")
)

cars_weight_class_df_Tamele70274.show(5, truncate=False)


# Exercise 2b (SQL)
We create "weight_classes" view and cross join with cars.
We filter by weight range and select Car, Weight, Class, Horsepower.


In [ ]:
weight_classes_Tamele70274.createOrReplaceTempView("weight_classes")

query_weight_sql_Tamele70274 = """
SELECT c.Car, c.Weight, w.Class, c.Horsepower
FROM cars c
CROSS JOIN weight_classes w
WHERE c.Weight >= w.MinWeight AND c.Weight < w.MaxWeight
"""

cars_weight_class_sql_Tamele70274 = spark_Tamele70274.sql(query_weight_sql_Tamele70274)
cars_weight_class_sql_Tamele70274.show(5, truncate=False)


# Exercise 3a (DF)
We compute mean, median, and standard deviation for MPG, Horsepower, and Weight.
We group by Origin and round results to two decimals.


In [ ]:
from pyspark.sql import functions as F

stats_df_Tamele70274 = (
    df_Tamele70274.groupBy("Origin")
    .agg(
        F.round(F.mean("MPG"),2).alias("mean_mpg"),
        F.round(F.expr("percentile(MPG, 0.5)"),2).alias("median_mpg"),
        F.round(F.stddev("MPG"),2).alias("std_mpg"),
        F.round(F.mean("Weight"),2).alias("mean_weight"),
        F.round(F.expr("percentile(Weight, 0.5)"),2).alias("median_weight"),
        F.round(F.stddev("Weight"),2).alias("std_weight"),
        F.count("*").alias("car_count")
    )
)

stats_df_Tamele70274.show(truncate=False)


# Exercise 3b (SQL)
We compute the same statistics using SQL functions.


In [ ]:
query_stats_sql_Tamele70274 = """
SELECT Origin,
       ROUND(AVG(MPG),2) AS mean_mpg,
       ROUND(percentile(MPG,0.5),2) AS median_mpg,
       ROUND(stddev(MPG),2) AS std_mpg,
       ROUND(AVG(Weight),2) AS mean_weight,
       ROUND(percentile(Weight,0.5),2) AS median_weight,
       ROUND(stddev(Weight),2) AS std_weight,
       COUNT(*) AS car_count
FROM cars
GROUP BY Origin
"""
spark_Tamele70274.sql(query_stats_sql_Tamele70274).show(truncate=False)


# Exercise 4a (DF)
We list distinct values in the Cylinders column.


In [ ]:
df_Tamele70274.select("Cylinders").distinct().orderBy("Cylinders").show()


# Exercise 4b (SQL)
We query distinct values of Cylinders using SQL.


In [ ]:
spark_Tamele70274.sql("SELECT DISTINCT Cylinders FROM cars ORDER BY Cylinders").show()


# Exercise 5a (DF)
We calculate min, max, and average MPG for cars with 4, 6, or 8 cylinders.


In [ ]:
df_Tamele70274.filter(df_Tamele70274.Cylinders.isin([4,6,8])) \
    .groupBy("Cylinders") \
    .agg(
        F.min("MPG").alias("min_mpg"),
        F.max("MPG").alias("max_mpg"),
        F.round(F.avg("MPG"),2).alias("avg_mpg")
    ).show()


# Exercise 5b (SQL)
We compute min, max, and average MPG for cars with 4, 6, or 8 cylinders using SQL.


In [ ]:
query_mpg_sql_Tamele70274 = """
SELECT Cylinders,
       MIN(MPG) AS min_mpg,
       MAX(MPG) AS max_mpg,
       ROUND(AVG(MPG),2) AS avg_mpg
FROM cars
WHERE Cylinders IN (4,6,8)
GROUP BY Cylinders
"""
spark_Tamele70274.sql(query_mpg_sql_Tamele70274).show()


# Exercise 6a (DF)
We select cars with MPG == 0.


In [ ]:
df_Tamele70274.filter(df_Tamele70274.MPG == 0).show(truncate=False)


# Exercise 6b (SQL)
We query cars with MPG == 0 using SQL.


In [ ]:
spark_Tamele70274.sql("SELECT * FROM cars WHERE MPG = 0").show(truncate=False)


# Exercise 7a (DF)
We select cars with MPG > 46.


In [ ]:
df_Tamele70274.filter(df_Tamele70274.MPG > 46).show(truncate=False)


# Exercise 7b (SQL)
We query cars with MPG > 46 using SQL.


In [ ]:
spark_Tamele70274.sql("SELECT * FROM cars WHERE MPG > 46").show(truncate=False)


# Exercise 7
We define lambda functions: double, is_even, polynomial_1, polynomial_2.
We test them with given inputs.


In [ ]:
double = lambda x: 2*x
is_even = lambda x: x % 2 == 0
poly1 = lambda a,b,c,d: a + b*c - d
poly2 = lambda arr, k: [x*k for x in arr]

print(double(5))
print(is_even(5))
print(poly1(1,2,3,4))
print(poly2([1,2,3],4))

ID = 70274
pars = [int(d) for d in f"{ID}"]
print(pars)
print(poly2(pars, ID % 10))


# Exercise 8
We use map() and filter() with lambda functions.


In [ ]:
number_list = [1,2,3,4]
print(list(map(lambda x: 2*x, number_list)))
print(list(filter(lambda x: x % 2 == 0, number_list)))


# Exercise 9
We create SparkContext, parallelize a list, and explore RDD operations.


In [ ]:
from pyspark import SparkContext
sc_Tamele70274 = SparkContext.getOrCreate()

rdd_Tamele70274 = sc_Tamele70274.parallelize([1,2,3,4,5])
print(rdd_Tamele70274.collect())
print(rdd_Tamele70274.take(3))
print(rdd_Tamele70274.first())


# Exercise 9 (textFile)
We create RDD from 'shakespeare.txt', print number of lines, first line, first three lines.


In [ ]:
baseRDD_Tamele70274 = sc_Tamele70274.textFile("shakespeare.txt")
print("Number of lines:", baseRDD_Tamele70274.count())
print("First line:", baseRDD_Tamele70274.first())
print("First three lines:", baseRDD_Tamele70274.take(3))


# Exercise 10
We use map() to split lines into lists of words.


In [ ]:
splitRDD1_Tamele70274 = baseRDD_Tamele70274.map(lambda line: line.split())
print("Number of elements:", splitRDD1_Tamele70274.count())
print("First element:", splitRDD1_Tamele70274.first())
print("First five elements:", splitRDD1_Tamele70274.take(5))


# Exercise 11
We use flatMap() to split lines into individual words.


In [ ]:
splitRDD2_Tamele70274 = baseRDD_Tamele70274.flatMap(lambda line: line.split())
print("Number of elements:", splitRDD2_Tamele70274.count())
print("First element:", splitRDD2_Tamele70274.first())
print("First five elements:", splitRDD2_Tamele70274.take(5))


# Exercise 12
We use flatMap() to split lines into words.
We print number of elements, first element, and first five elements.
Later, we will remove stopwords to reduce the dataset.


In [ ]:

splitRDD2_Tamele70274 = baseRDD_Tamele70274.flatMap(lambda line: line.split())

print("Number of elements:", splitRDD2_Tamele70274.count())
print("First element:", splitRDD2_Tamele70274.first())
print("First five elements:", splitRDD2_Tamele70274.take(5))


# Exercise 13
We import NLTK stopwords, download them, and print their length and first ten words.


In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words_Tamele70274 = stopwords.words('english')
print("Length of stop_words:", len(stop_words_Tamele70274))
print("First ten stopwords:", stop_words_Tamele70274[:10])


# Exercise 14
We filter splitRDD2 to remove stopwords.
We print the length of the filtered RDD and the first ten words.


In [ ]:
splitRDD_nostop_Tamele70274 = splitRDD2_Tamele70274.filter(lambda word: word.lower() not in stop_words_Tamele70274)

print("Length of splitRDD_nostop:", splitRDD_nostop_Tamele70274.count())
print("First ten words (no stopwords):", splitRDD_nostop_Tamele70274.take(10))
